\newpage

# OBJETIVOS

- Analizar el comportamiento histórico del AQI en las 4 regiones definido por el Censo de Estados Unidos para identificar tendencias y variaciones de la calidad del aire, mediante el uso de los datos disponibles.

- Aplicar el método de Splines Cúbicos con frontera natural para interpolar y modelar la evolución histórica del AQI en las regiones definidas por el Censo de los Estados Unidos.

- Utilizar el método de Mínimos Cuadrados para obtener un modelo de ajuste que permita realizar extrapolaciones aproximadas de la tendencia futura del AQI en cada región. 

# MARCO TEÓRICO

## Índice de la calidad del aire (AQI)

Este proyecto busca analizar el ├ìndice de Calidad del Aire (AQI) que es una herramienta desarrollada por la Agencia de Protección Ambiental de los Estados Unidos (EPA) con el objetivo de comunicar de manera sencilla el nivel de contaminación del aire y su posible impacto en la salud de la población. El AQI se divide en seis categorías principales, cada una asociada a un nivel de riesgo para la salud: buena (0–50), moderada (51–100), no saludable para grupos sensibles (101–150), no saludable (151–200), muy no saludable (201–300) y peligrosa (301 o más) [@epa2023aqi].

## Contaminantes atmosféricos

El cálculo del AQI se basa en la medición de diferentes contaminantes atmosféricos regulados por la EPA, los que incluyen el ozono (O3), monóxido de carbono (CO), dióxido de nitrógeno (NO2), dióxido de azufre (SO2) y material particulado como PM2.5 Y PM10. Cada uno de estos contaminantes tiene efectos sobre la salud humana, siendo que la exposición prolongada a estos contaminantes puede generar enfermedades respiratorias, agravar el asma, disminuir la función pulmonar e incrementar el riesgo de enfermedades cardiovasculares [@epa2023aqi]. Por esta razón, resulta fundamental el monitoreo de la calidad del aire para la salud de la población.

## Interpolación mediante Splinés Cúbicos

En el contexto del análisis de datos históricos del periodo 1980–2021, se emplean los splines cúbicos naturales como método de interpolación por tramos. Este método permite aproximar el comportamiento del AQI mediante polinomios de tercer grado entre nodos consecutivos, garantizando suavidad en la representación. En particular, el spline cúbico natural impone condiciones de frontera libres, asumiendo que la segunda derivada en los extremos del intervalo es igual a cero, lo que permite modelar de forma continua la evolución del AQI a lo largo del tiempo [@splinesCubicos]. Su aplicación será para la interpolación de los datos disponibles para analizar la variación del AQI dentro del intervalo observado.

## Extrapolación mediante Mínimos Cuadrados

Por otro lado, el método de mínimos cuadrados es una técnica de regresión que permite obtener una función de mejor ajuste a partir de un conjunto de datos, minimizando la suma de los residuos al cuadrado. Una vez obtenido el modelo, este se puede utilizar para realizar extrapolaciones [@monashLeastSquares]. Dentro del proyecto se usá para hacer extrapolaciones sobre el comportamiento del AQI, ya que las estimaciones se realizaran fuera del rango de los datos observados.

Para realizar el ajuste polinómico mediante mínimos cuadrados, los datos pueden representarse como un sistema lineal **Ac = y**, donde la matriz A corresponde a una matriz de Vandermonde. En este proyecto, la matriz de Vandermonde se construye utilizando los años como variable independiente y el Median AQI como variable dependiente, permitiendo generar un modelo polinómico para representar la tendencia histórica y realizar extrapolaciones futuras [@ajustepolinomico]. Para evitar inestabilidad numérica en los coeficientes al trabajar con grados polinómicos altos, los años se centran respecto a un año de referencia antes de construir la matriz de Vandermonde, técnica que reduce el mal condicionamiento del sistema causado por la escala de la variable independiente, sin que esto afecte la forma del modelo ajustado [@Centering].

Sin embargo, al momento de realizar la extrapolación se debe tener precaución, ya que la relación entre las variables puede no mantenerse fuera del intervalo. Por ello, las extrapolaciones obtenidas para cada región se basarán en el grado polinómico que mejor se acerque a los datos reales durante la etapa de validación.

# PRERREQUISITOS

- **Dataset 1980-2021 Yearly Air Quality Index from the EPA:** Se trabajará con este dataset que contiene registros anuales relacionados con la calidad del aire en distintos estados y condados de Estados Unidos durante el periodo 1980–2021. Los datos fueron obtenidos a través de Kaggle [@wadkins2021aqi].

Durante la etapa de limpieza de datos se seleccionará el periodo entre 1980 y 2020. Esto es debido que hasta 2020, todos los registros mostraban información de los condados requeridos para el análisis, sin embargo en 2021, varios condados no cuentan con información registrada.

- **NumPy:** Se utiliza esta libreria para realizar cálculos numéricos como la consutricción de la matriz de Vandermonde o resolución de mínimos cuadrados.

- **Pandas:** Se utiliza para la lectura, limpieza y agrupación de los datos del dataset.

- **Matplotlib:** Se utiliza para la generación de gráficas de la interpolación de splines cúbicos y extrapolación de mínimos cuadrados.

# DESARROLLO

## Descripción del dataset

El dataset usado contiene los registros anuales en relación a la calidad del aire en Estados Unidos durante el periodo entre 1980 y 2021.

El dataset incluye las siguientes variables:
-	**State y County:** Indican la ubicación geográfica de los registros.
-	**Latitude y Longitude:** Indican las coordenadas geográficas del condado.
-	**Year:** Representa el año en los que se realizaron los registros.
-	**Median AQI:** Representa un valor mediano del AQI que se registró durante el año. Será una de las variables principales para analizar la tendencia de la calidad del aire. 
-	**90th Percentile AQI:** Indica el valor máximo del AQI alcanzado en el 90% de los días del año, excluyendo el 10% que representaría los días más contaminados.
-	**Max AQI:** Indica el valor máximo del AQI registrado en el año. 
-	**Days with AQI:** Indica la cantidad de días con registros de calidad del aire disponibles. 
-	**Good Days, Moderate Days, Unhealthy for Sensitive Groups Days, Unhealthy Days, Very Unhealthy Days y Hazardous Days:** Representan la cantidad de días en el año en que la calidad del aire se encontró dentro de cada rango del índice AQI, clasificando según el nivel de riesgo para la salud.
-	**Days CO, Days NO2, Days Ozone, Days SO2, Days PM2.5 y Days PM10:** Representan la cantidad de días en que cada contaminante fue el principal responsable del nivel de AQI en ese día. 

Durante la primera etapa de preprocesamiento se limpiaron los datos de la base y se agruparon los registros por regiones geográficas para facilitar el análisis comparativo.

## Variables utilizadas

Las siguientes variables fueron consideradas para su uso y se encuentran en el archivo que contiene el dataset normalizado:

| Variables | Descripción |
|---|---|
| Year | Año del registro |
| Region | Region geográfica asignada por el Censo |
| Median AQI | Mediana del AQI |
| AQI level days | Número de días clasificados según el nivel de la calidad del aire (Good, Moderate, Unhealthy, Hazardous) |
| Pollutant days | Número de dias de según el contaminante principal (CO, NO2, O3, SO2, PM2.5, PM10) |

## Fases de desarrollo

### Fase 1: Preparación de datos

En esta fase se realizó la limpieza, normalización y agrupación del dataset. Primero, se asignó una región geográfica a cada registro mediante la función **asignar_region()**, la cual clasifica cada estado de Estados Unidos dentro de una de las cuatro regiones definidas por el Censo: Northeast, Midwest, South y West.

Luego, mediante la función **limpiar_agrupar()**, se cargó el dataset original, se eliminaron los registros con valores nulos en las variables seleccionadas y se descartaron aquellos que no pertenecían a una región. Además, se calcularon las proporciones correspondientes a las categorías de los días del AQI y a los contaminantes principales. También, se seleccionaron únicamente los datos entre el periodo de 1980 y 2020, debido a que el año 2021 no presenta registros de todos los condados. Finalmente, los datos se agruparon por región y año.

Como resultado de esta etapa se obtuvo un dataset limpio y normalizado, utilizado como entrada para los métodos numéricos implementados en las siguientes fases.

### Fase 2: Aplicación de Splines Cúbicos

En esta fase se implementó el método de interpolación mediante splines cúbicos naturales utilizando la función **spline_cubico_natural()**, encargada de calcular los coeficientes de los polinomios cúbicos para cada intervalo de años.

Se seleccionó el spline cúbico natural debido a que el análisis se limita entre le período de 1980 a 2020 y no se cuenta con información suficiente para establecer condiciones específicas sobre la variación del AQI en los extremos del intervalo. Por lo tanto, la frontera natural permite obtener una interpolación suave sin imponer comportamientos adicionales en los extremos [@splinesCubicos].

Luego, la función **evaluarSpline()** permite calcular el valor del spline para cualquier año comprendido dentro del intervalo de datos, obteniendo una representación continua del comportamiento histórico del Median AQI. Ádemas se implemento la función **graficar_spline_por_region()** que se utiliza los coeficientes calculados para generar la representación gráfica de la interpolación correspondiente a cada región.

Finalmente, la función **valores_predominantes()** muestra para cada región y por año, el AQI promedio, la categoría de día predominante y el contaminante principal asociado al nivel de AQI, con el propósito de analizar la tendencia histórica que se ve reflejada en el spline.

### Fase 3: Aplicación de Mínimos Cuadrados

En esta etapa se implementó el método de mínimos cuadrados mediante la función **minimos_cuadrados()**, la cual construye la matriz de Vandermonde utilizando los años del dataset respecto a un año de referencia para evitar inestabilidad numérica [@Centering], y calcula los coeficientes del polinomio ajustado mediante el algoritmo de mínimos cuadrados.

A partir de este ajuste, la función **validar_modelo()** permite comparar distintos grados polinómicos por región, dividiendo los datos históricos en un conjunto de entrenamiento y uno de prueba según un año límite definido. Para cada grado especificado, se ajusta el polinomio sobre los datos de entrenamiento y se representa gráficamente junto con los datos reales, permitiendo comparar visualmente qué grado sigue de forma más adecuada la tendencia histórica del Median AQI en cada región.

Este procedimiento de validación permite seleccionar para cada región, el grado que mejor represente el comportamiento observado, cuyo dato se va a utilizar para realizar las extrapolaciones.

### Fase 4: Extrapolación del modelo

En esta fase se realiza la extrapolación del Median AQI para cada región utilizando el grado anteriormente identificado.

La función **mostrar_valores_extrapolacion()** genera una tabla con los valores extrapolados del Median AQI para los años futuros definidos por el usuario, ajustando el polinomio correspondiente a cada región mediante **minimos_cuadrados()** sobre los años respecto al último año disponible en el dataset.

Además, la función **graficar_extrapolacion_por_region()** representa gráficamente, para cada región, los datos históricos junto con la curva de extrapolación obtenida a partir del grado óptimo e incluyendo la ecuación del modelo.

### Fase 5: Interpretación de los resultados 

Finalmente, se realiza un análisis comparativo entre las cuatro regiones a partir de las gráficas, con el propósito de interpretar el comportamiento del AQI en cada una de ellas.

A partir de estos resultados, se busca identificar:

- Las tendencias del AQI a lo largo del período analizado.
- Los contaminantes que predominan en cada una de las regiones.
- Las diferencias en la calidad del aire entre las distintas regiones.
- El comportamiento futuro proyectado del AQI a partir de los modelos de extrapolación.

## Etapa de análisis

El proceso para la preparación de los datos se realizó en las siguientes fases:

### 1. Carga y filtración de los datos
Se cargó el dataset original en formato CSV y se filtraron únicamente los registros del periodo 1980-2020, descartando el año 2021 debido a que la mayoria de condados no presentaban registro durante ese año.

### 2. Eliminación de valores nulos
Se eliminaron todas las filas que presentaran valores faltantes en las variables a utilizar para garantizar la integridad del análisis numérico.

### 3. Asignación de regiones geográficas
Mediante la función asignar_region() se clasificó cada estado dentro de su región geográfica correspondiente según las divisiones del Censo de Estados Unidos (Northeast, Midwest, South y West). Los registros que no pertenecían a ninguna región, como Puerto Rico y Baja California, fueron descartados.

### 4. Conversión a porcentajes
Las variables de días por categoría **(Good Days, Moderate Days, Unhealthy for Sensitive Groups Days, Unhealthy Days, Very Unhealthy Days, Hazardous Days)** y las variables de contaminantes **(Days CO, Days NO2, Days Ozone, Days SO2, Days PM2.5, Days PM10)** se normalizaron dividiendo cada valor entre el total de días registrados ese año, obteniendo una proporción entre 0 y 1. Esto se realizó debido a que la cantidad de días registrados variaba entre los distintos años, lo que podía generar comparaciones injustas. Al utilizar estas proporciones, se representa de mejor manera el peso relativo de cada variable, permitiendo realizar comparaciones consistentes entre regiones.

### 5. Agrupación por región y año
Se agruparon todos los condados de cada región por año calculando el promedio de todas las variables numéricas, obteniendo un único valor por región por año. Esto redujo el dataset de 34,187 registros a 164 registros.

## Métodos númericos implementados

### Spline Cúbico con frontera natural

Los splines cúbicos naturales fueron utilizados como método de interpolación para modelar la variación histórica del AQI en las cuatro regiones definidas por el Censo de Estados Unidos. Este método permite aproximar una función mediante una serie de polinomios cúbicos definidos en intervalos consecutivos entre los datos disponibles.

En este proyecto, los puntos utilizados para la interpolación corresponden a los pares:

$$
(x,y)=(Year, Median\ AQI)
$$

Para cada intervalo entre dos años consecutivos se define un polinomio cúbico:

$$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3
$$

donde los coeficientes determinan la forma del polinomio dentro del intervalo correspondiente.

La construcción del spline requiere que los polinomios cumplan condiciones de continuidad entre intervalos, garantizando que la función sea continua. Además, al utilizar un spline cúbico natural se establecen condiciones de frontera:

$$
S''(x_0)=0,\qquad S''(x_n)=0
$$

Estas condiciones permiten evitar suposiciones sobre el comportamiento de la función fuera del rango observado, generando una curva suave dentro del intervalo de datos disponible [@splinesCubicos].

## Mínimos Cuadrados

El método de mínimos cuadrados fue utilizado para realizar la extrapolación de la tendencia futura del AQI por región. Este método permite obtener un polinomio que representa la relación aproximada entre los años registrados y el valor del Median AQI.

El problema se plantea como:

$$
Ac = y
$$

donde:

**A** es la matriz de Vandermonde construida con los valores del año.  
**c** representa los coeficientes desconocidos.  
**y** representa los valores históricos del Median AQI.

Para un polinomio de grado n, la matriz de Vandermonde se construye utilizando los valores de la variable independiente **x**, en este caso los años registrados:

$$
A=
\begin{bmatrix}
x_1^n & x_1^{n-1} & ... & x_1 & 1\\
x_2^n & x_2^{n-1} & ... & x_2 & 1\\
\vdots & \vdots & \ddots & \vdots & \vdots\\
x_m^n & x_m^{n-1} & ... & x_m & 1
\end{bmatrix}
$$

donde cada fila representa un registro histórico del AQI y cada columna corresponde a un término del polinomio.

La matriz de Vandermonde permite representar un polinomio de grado n:

$$
P(x)=a_nx^n+a_{n-1}x^{n-1}+...+a_1x+a_0
$$

El vector de coeficientes del polinomio se representa como:

$$
c=
\begin{bmatrix}
a_n\\
a_{n-1}\\
\vdots\\
a_1\\
a_0
\end{bmatrix}
$$

Debido a que existen más ecuaciones que incógnitas, se busca una aproximación que minimice la diferencia entre los valores observados y los valores obtenidos mediante el polinomio.

Para obtener esta solución se multiplica el sistema original por la matriz transpuesta de Vandermonde A^T:

$$
A^T(Ac)=A^Ty
$$

Aplicando la propiedad asociativa de la multiplicación matricial se obtiene:

$$
(A^TA)c=A^Ty
$$

Este sistema corresponde a las ecuaciones del método de mínimos cuadrados [@ajustepolinomico]. Con este sistema ya es posible obtener los coeficientes del polinomio.

## Funciones implementadas

**asignar_region(estado)**  
Asigna a cada estado de Estados Unidos la región geográfica del Censo **(Northeast, Midwest, South o West)** según el diccionario **regiones**. Si el estado no pertenece a ninguna de ellas, devuelve **Sin Región**.

**limpiar_agrupar()**  
Lee el dataset original, filtra los datos entre 1980 y 2020, elimina los registros con valores faltantes y asigna la región correspondiente a cada estado. Luego convierte las variables de días y contaminantes en proporciones respecto al total de días registrados, agrupa los datos por región y año calculando el promedio y guarda el dataset limpio.

**spline_cubico_natural(n, x, a)**  
Calcula los coeficientes de un spline cúbico natural a partir de los nodos **x** y sus valores **a**, resolviendo el sistema tridiagonal con condiciones de frontera naturales.

**evaluarSpline(x_eval, x, coeficientes)**  
Evalúa el spline cúbico natural en un valor **x_eval**, identifica el intervalo correspondiente y calcula el valor interpolado con los coeficientes obtenidos.

**graficar_spline_por_region(df)**  
Genera la gráfica del **Median AQI** para cada región usando splines cúbicos naturales y la muestra junto con los datos originales. Cada gráfico se guarda como una imagen.

**valores_predominantes(df)**  
Muestra para cada región y año el **Median AQI** promedio, la categoría de calidad del aire predominante y el contaminante principal, junto con el porcentaje de días que representa cada uno.

**minimos_cuadrados(x, y, grado)**  
Ajusta un polinomio del grado indicado mediante mínimos cuadrados, construyendo la matriz de Vandermonde y resolviendo el sistema con **numpy.linalg.lstsq**. Devuelve los coeficientes y la ecuación del modelo.

**validar_modelo(df, grados, año_limite)**  
Compara distintos grados de polinomios para ajustar el **Median AQI** por región. Divide los datos en entrenamiento y prueba según un año límite y grafica los resultados para facilitar la comparación.

**mostrar_valores_extrapolacion(df, mejores_grados, año_fin)**  
Genera una tabla con los valores extrapolados del **Median AQI** para cada región utilizando el grado polinómico seleccionado y proyectando los resultados hasta el año indicado.

**graficar_extrapolacion_por_region(df, mejores_grados, año_fin)**  
Grafica los datos históricos junto con la curva de extrapolación obtenida con el grado polinómico seleccionado. Además, muestra la ecuación del modelo y marca el inicio del período de extrapolación.

## Flujograma

## Enlace de Github

https://github.com/DRKca089/Grupo2_SaludPublica.git

## Estadísticas del repositorio

El desarrollo del proyecto se realizó en un repositorio compartido de GitHub, en el cual todos los integrantes participaron como colaboradores.  

A continuación, se observan las estadísticas del repositorio y colaboradores:

- **JUSTAILS:** Justin Guevara  
- **DRKca089:** Derek Cahuate  

![Gráfica de commits realizados al repositorio](./inFiles/Estadisticas_repositorio.png)

## RESULTADOS FINALES

1. **Interpolación mediante splines cúbicos**

Como se observa en las Figuras 1, 2, 3 y 4, la interpolación mediante Splines Cúbicos permitió representar de forma continua el comportamiento histórico del Median AQI entre 1980 a 2020. En todas las regiones se observa una tendencia general de disminución del AQI a lo largo del periodo, lo que indicaria una mejora en la calidad del aire a nivel nacional.

![Interpolación del Median AQI para la región Midwest.](../graficas/interpolacion_Midwest.png)

En la Figura 1 se observa una tendencia decreciente desde 1980 hasta aproximadamente 1995. Durante gran parte de este período, el dióxido de azufre (SO2) fue el contaminante predominante. Entre 1995 y 2015 el AQI presenta un comportamiento estable, pero posteriormente, se vuelve a mostrar una tendencia decreciente hacia el final del período analizado.

![Interpolación del Median AQI para la región Northeast.](../graficas/interpolacion_Northeast.png)

En la Figura 2 se observa que esta región inició el período con los valores más altos de AQI. Sin embargo, a lo largo de los años, la tendencia comienza a decrecer. Este comportamiento podría estar asociado al cambio del contaminante predominante, pasando del dióxido de azufre (SO2) durante los primeros años al ozono (O3) a partir de finales de la década de 1980.

![Interpolación del Median AQI para la región South.](../graficas/interpolacion_South.png)

En la Figura 3 se observa un comportamiento más irregular, con un incremento del AQI cercano al año 2000. Posteriormente, la tendencia presenta una disminución hasta el final del período analizado. Durante todo el período analizado, el ozono (O3) fue el principal contaminante responsable del nivel del AQI.

![Interpolación del Median AQI para la región West.](../graficas/interpolacion_West.png)

En la Figura 4 se observa una disminución del nivel del AQI entre 1980 y 1995. Sin embargo, en los años posteriores la tendencia presenta ciertas irregularidades frente a las demás regiones. Este comportamiento coincide con un incremento del PM2.5 como contaminante principal durante los últimos años del período.

2. **Validación del modelo de Mínimos Cuadrados**

Para evaluar la capacidad de extrapolación del modelo, se dividieron los datos en un conjunto de entrenamiento que abarca los años entre 1980 y 2012, y un conjunto de validación que va desde 2013 hasta 2020. Para cada región se ajustaron y graficaron 4 grados polinómicos distintos utilizando el conjunto de entrenamiento, con el fin de comparar visualmente cuál se ajusta mejor a los datos y seleccionar el grado más adecuado para las extrapolaciones.

![Validación del ajuste para la región Midwest.](../graficas/validacion_Midwest.png)

En la Figura 5 el polinomio de grado 1 es el que mejor sigue el comportamiento de los datos de validación.

![Validación del ajuste para la región Northeast.](../graficas/validacion_Northeast.png)

Para la Figura 6 el grado 3 presenta el ajuste más cercano a los puntos reales para la validación.

![Validación del ajuste para la región South.](../graficas/validacion_South.png)

Para la Figura 7 muestra que el grado 3 es el que mejor se aproxima a la tendencia en comparación de los demás grados.

![Validación del ajuste para la región West.](../graficas/validacion_West.png)

Para la Figura 8 se observa que el grado 4 es el que más se acerca a los datos reales en comparación de los otros tipo de grados.

3. **Extrapolación del modelo de Mínimos Cuadrados**

Para la parte de extrapolación de datos, se utilizó el grado polinómico obtenido durante la validación para cada una de las regiones, dando como resultado los siguientes modelos:

**Midwest** (grado 1):
$$
y = -0.271x + 36.752
$$

**Northeast** (grado 3):
$$
y = -0.001x^3 - 0.029x^2 - 0.247x + 39.887
$$

**South** (grado 3):
$$
y = -0.002x^3 - 0.091x^2 - 0.962x + 40.649
$$

**West** (grado 4):
$$
y = -0.000x^4 - 0.004x^3 - 0.059x^2 - 0.019x + 36.150
$$

A partir de estos modelos, se decidió extrapolar el Median AQI desde el año 2021 hasta el año 2030 para cada región.

![Extrapolación para la región Midwest.](../graficas/extrapolacion_Midwest.png)

En la Figura 9 se obtuvieron los siguientes valores extrapolados para la región Midwest:

| Año  | Median AQI |
|------|------------|
| 2021 | 33.93      |
| 2022 | 33.65      |
| 2023 | 33.37      |
| 2024 | 33.08      |
| 2025 | 32.80      |
| 2026 | 32.51      |
| 2027 | 32.23      |
| 2028 | 31.94      |
| 2029 | 31.66      |
| 2030 | 31.38      |

![Extrapolación para la región Northeast.](../graficas/extrapolacion_Northeast.png)


En la Figura 10 se obtuvieron los siguientes valores extrapolados para la región Northeast:

| Año  | Median AQI |
|------|------------|
| 2021 | 33.73      |
| 2022 | 32.53      |
| 2023 | 31.19      |
| 2024 | 29.71      |
| 2025 | 28.08      |
| 2026 | 26.29      |
| 2027 | 24.33      |
| 2028 | 22.19      |
| 2029 | 19.88      |
| 2030 | 17.37      |

![Extrapolación para la región South.](../graficas/extrapolacion_South.png)

En la Figura 11 se obtuvieron los siguientes valores extrapolados para la región South:

| Año  | Median AQI |
|------|------------|
| 2021 | 29.26      |
| 2022 | 27.03      |
| 2023 | 24.60      |
| 2024 | 21.94      |
| 2025 | 19.05      |
| 2026 | 15.92      |
| 2027 | 12.54      |
| 2028 | 8.90       |
| 2029 | 5.00       |
| 2030 | 0.82       |

![Extrapolación para la región West.](../graficas/extrapolacion_West.png)

En la Figura 12 se obtuvieron los siguientes valores extrapolados para la región West:

| Año  | Median AQI |
|------|------------|
| 2021 | 32.15      |
| 2022 | 30.60      |
| 2023 | 28.74      |
| 2024 | 26.53      |
| 2025 | 23.95      |
| 2026 | 20.96      |
| 2027 | 17.53      |
| 2028 | 13.63      |
| 2029 | 9.22       |
| 2030 | 4.27       |

# CONCLUSIONES

- En conclusión, durante el período entre 1980 y 2020, la calidad del aire mejoró en todas las regiones de Estados Unidos, lo cual se evidencia en la disminución del AQI promedio y en el aumento del porcentaje de días clasificados como buenos. Además, se observa un cambio en el contaminante principal responsable del AQI. En las regiones Midwest y Northeast, el dióxido de azufre (SO2) fue reemplazado por el ozono (O3). En la región South, el ozono se mantuvo como el contaminante principal durante todo el período analizado.  Mientras que la región West mostró un comportamiento diferente, variando entre el ozono y el PM10, pero en los últimos años ha comenzado a predominar el PM2.5.

- La interpolación mediante splines cúbicos naturales permitió representar la evolución del AQI entre 1980 y 2020 en cada región, mostrando de forma continua el comportamiento de los datos a lo largo del tiempo. De esta forma, resulta ser eficiente para observar la tendencia dentro del intervalo seleccionado.

- La extrapolación mediante mínimos cuadrados, junto con su validación, permitió evaluar el desempeño del modelo para realizar extrapolaciones. Para ello, se centraron los años con el fin de obtener un mejor ajuste numérico. Como resultado, cada región presentó un grado polinómico que se adaptó mejor a los datos: grado 4 para West, grado 1 para Midwest y grado 3 para Northeast y South. Estos modelos fueron utilizados para realizar las respectivas extrapolaciones. Sin embargo, es importante destacar que una extrapolación no es una predicción exacta, sino una estimación basada en la tendencia actual de los datos.

# RECOMENDACIONES

- Para futuras extrapolaciones, se recomienda utilizar polinomios de bajo grado o comparar con otros modelos, ya que los polinomios de grado alto pueden generar sobreajuste y producir extrapolaciones poco realistas.

- Además, se recomendaria desarrollar modelos que permitan estimar los posibles contaminantes principales que determinen el nivel del AQI y la categoría de calidad del aire predominante en los próximos años.

- Se recomienda ampliar el estudio incorporando otras variables ambientales, como incendios forestales, temperatura y velocidad del viento, con el fin de analizar mejor la relación entre las variaciones del AQI. 

# REFERENCIAS